# 03 — Gráficos das variáveis candidatas vs. alvo

## Objetivo

Mostrar visualmente que os atributos selecionados no notebook 02 têm relação com a variável-alvo (FALTA x EXCESSO de chuva). Cada gráfico compara o comportamento de uma ou mais variáveis entre as duas classes.

### 1. Carregamento e preparação dos dados

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ARQUIVO_BASE = (
    ROOT / "data" / "raw"
    / "BD_Atlas_1991_2025_v1.1_2026.08.06_Consolidado.xlsx"
)

DIR_FIGURAS = ROOT / "reports" / "figures"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

DIR_METRICAS = ROOT / "reports" / "metrics"
DIR_METRICAS.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(ARQUIVO_BASE, sheet_name="Atlas Valores Corrigidos")

# Mapeamento do alvo (mesmo do notebook 01, seção 8)
mapa_classes = {
    "Estiagem e Seca": "FALTA",
    "Enxurradas": "EXCESSO",
    "Inundações": "EXCESSO",
    "Alagamentos": "EXCESSO",
}

df["alvo"] = df["descricao_tipologia"].map(mapa_classes)
df = df[df["alvo"].notna()].copy()

print(f"Registros: {len(df):,}")
print()
display(df["alvo"].value_counts())

Registros: 51,921



alvo
FALTA      33493
EXCESSO    18428
Name: count, dtype: int64

### 1.1. Verificação das colunas disponíveis

In [2]:
print("Colunas do DataFrame:")
for i, col in enumerate(df.columns):
    print(f"  {i:02d}. {col}")

Colunas do DataFrame:
  00. Protocolo_S2iD
  01. Nome_Municipio
  02. Sigla_UF
  03. regiao
  04. Data_Registro
  05. Data_Evento
  06. Cod_Cobrade
  07. tipologia
  08. descricao_tipologia
  09. grupo_de_desastre
  10. Cod_IBGE_Mun
  11. Setores Censitários
  12. Status
  13. DH_Descricao
  14. DH_MORTOS
  15. DH_FERIDOS
  16. DH_ENFERMOS
  17. DH_DESABRIGADOS
  18. DH_DESALOJADOS
  19. DH_DESAPARECIDOS
  20. DH_AFETADOS_SECA_ESTIAGEM
  21. DH_total_danos_humanos_diretos
  22. DH_OUTROS AFETADOS
  23. DM_Descricao
  24. DM_Uni Habita Danificadas
  25. DM_Uni Habita Destruidas
  26. DM_Uni Habita Valor
  27. DM_Inst Saúde Danificadas
  28. DM_Inst Saúde Destruidas
  29. DM_Inst Saúde Valor
  30. DM_Inst Ensino Danificadas
  31. DM_Inst Ensino Destruidas
  32. DM_Inst Ensino Valor
  33. DM_Inst Serviços Danificadas
  34. DM_Inst Serviços Destruidas
  35. DM_Inst Serviços Valor
  36. DM_Inst Comuni Danificadas
  37. DM_Inst Comuni Destruidas
  38. DM_Inst Comuni Valor
  39. DM_Obras de I

### 1.2. Colunas de vazamento (excluídas dos gráficos)

As colunas abaixo revelam diretamente o tipo de desastre ou contêm o nome da tipologia. Elas não serão usadas como preditoras e não aparecem nos gráficos.

In [3]:
COLUNAS_VAZAMENTO = [
    "descricao_tipologia",
    "tipologia",
    "Cod_Cobrade",
    "Protocolo_S2iD",
    "grupo_de_desastre",
    "DH_AFETADOS_SECA_ESTIAGEM",
]

print("Colunas exclu\u00eddas por vazamento:")
for c in COLUNAS_VAZAMENTO:
    print(f"  - {c}")

Colunas excluídas por vazamento:
  - descricao_tipologia
  - tipologia
  - Cod_Cobrade
  - Protocolo_S2iD
  - grupo_de_desastre
  - DH_AFETADOS_SECA_ESTIAGEM


---

### 2. Gráfico 1: danos humanos e moradias

Percentual de registros com valor maior que zero para danos humanos (desabrigados, desalojados, óbitos, feridos) e unidades habitacionais danificadas/destruídas, separado por classe.

In [4]:
colunas_danos = [
    "DH_DESABRIGADOS",
    "DH_DESALOJADOS",
    "DH_MORTOS",
    "DH_FERIDOS",
    "DM_Uni Habita Danificadas",
    "DM_Uni Habita Destruidas",
]

rotulos_danos = [
    "Desabrigados",
    "Desalojados",
    "\u00d3bitos",
    "Feridos",
    "Moradias\ndanificadas",
    "Moradias\ndestr.",
]

dados_g1 = []
for col, rotulo in zip(colunas_danos, rotulos_danos):
    for classe in ["FALTA", "EXCESSO"]:
        sub = df[df["alvo"] == classe]
        pct = (sub[col] > 0).mean() * 100
        dados_g1.append({"Vari\u00e1vel": rotulo, "Classe": classe, "% > 0": pct})

df_g1 = pd.DataFrame(dados_g1)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_g1, x="Vari\u00e1vel", y="% > 0", hue="Classe", ax=ax)
ax.set_title("Registros com valor > 0: danos humanos e moradias")
ax.set_ylabel("% de registros com valor > 0")
ax.set_xlabel("")
ax.legend(title="Classe")

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=8, padding=2)

fig.tight_layout()
fig.savefig(DIR_FIGURAS / "grafico1_danos_humanos_moradias.png", dpi=150)
plt.show()
print(f"Salvo em {DIR_FIGURAS / 'grafico1_danos_humanos_moradias.png'}")

Salvo em C:\Users\Virtron\Documents\sergio pessoal\atlas-digital\reports\figures\grafico1_danos_humanos_moradias.png


C:\Users\Virtron\AppData\Local\Temp\ipykernel_18836\1125407844.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Leitura:** Desalojados aparecem em 44,5% dos registros de EXCESSO contra apenas 0,6% dos de FALTA (diferenca de 43,9 pp). Moradias danificadas seguem o mesmo padrao: 39,9% em EXCESSO vs. 0,1% em FALTA. Desabrigados: 31,8% vs. 0,2%. Obitos e feridos tambem sao mais frequentes em EXCESSO (4,3% e 10,9%), embora com diferencas menores. Em resumo, danos fisicos imediatos a pessoas e moradias sao tipicos de excesso de chuva e quase inexistentes em falta.

---

### 3. Gráfico 2: prejuízos econômicos (agricultura, pecuária, abastecimento)

Percentual de registros com valor maior que zero para prejuízos em agricultura, pecuária e abastecimento de água.

In [5]:
# Localizar colunas pelo prefixo para evitar problemas de encoding
def encontrar_coluna(df, prefixo):
    """Retorna a primeira coluna que comeca com o prefixo."""
    candidatas = [c for c in df.columns if c.startswith(prefixo)]
    if candidatas:
        return candidatas[0]
    raise KeyError(f"Nenhuma coluna com prefixo '{prefixo}'")

col_agricultura = encontrar_coluna(df, "PEPR_Agricultura")
col_pecuaria = encontrar_coluna(df, "PEPR_Pecu")
col_abast = encontrar_coluna(df, "PEPL_Abast")

colunas_prejuizos = [col_agricultura, col_pecuaria, col_abast]
rotulos_prejuizos = ["Agricultura", "Pecu\u00e1ria", "Abastecimento\nde \u00e1gua"]

dados_g2 = []
for col, rotulo in zip(colunas_prejuizos, rotulos_prejuizos):
    for classe in ["FALTA", "EXCESSO"]:
        sub = df[df["alvo"] == classe]
        pct = (sub[col].fillna(0) > 0).mean() * 100
        dados_g2.append({"Vari\u00e1vel": rotulo, "Classe": classe, "% > 0": pct})

df_g2 = pd.DataFrame(dados_g2)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=df_g2, x="Vari\u00e1vel", y="% > 0", hue="Classe", ax=ax)
ax.set_title("Registros com valor > 0: preju\u00edzos econ\u00f4micos")
ax.set_ylabel("% de registros com valor > 0")
ax.set_xlabel("")
ax.legend(title="Classe")

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=8, padding=2)

fig.tight_layout()
fig.savefig(DIR_FIGURAS / "grafico2_prejuizos_economicos.png", dpi=150)
plt.show()
print(f"Salvo em {DIR_FIGURAS / 'grafico2_prejuizos_economicos.png'}")

Salvo em C:\Users\Virtron\Documents\sergio pessoal\atlas-digital\reports\figures\grafico2_prejuizos_economicos.png


C:\Users\Virtron\AppData\Local\Temp\ipykernel_18836\2615508638.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Leitura:** Agricultura tem prejuizo registrado em 58,4% dos eventos de FALTA contra 32,6% em EXCESSO (diferenca de 25,7 pp). Pecuaria segue o mesmo padrao: 45,3% vs. 21,6% (23,8 pp). Abastecimento de agua apresenta a maior diferenca do grupo: 44,6% em FALTA vs. 14,0% em EXCESSO (30,6 pp). Faz sentido: seca e estiagem impactam diretamente a producao rural e o abastecimento hidrico.

---

### 4. Gráfico 3: quantidade de registros por mês do evento

Distribuição temporal dos desastres ao longo dos meses do ano, separada por classe.

In [6]:
df["mes_evento"] = pd.to_datetime(df["Data_Evento"]).dt.month

contagem_mes = (
    df.groupby(["mes_evento", "alvo"])
    .size()
    .reset_index(name="quantidade")
)

nomes_meses = {
    1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr",
    5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago",
    9: "Set", 10: "Out", 11: "Nov", 12: "Dez",
}
contagem_mes["mes_nome"] = contagem_mes["mes_evento"].map(nomes_meses)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=contagem_mes,
    x="mes_nome",
    y="quantidade",
    hue="alvo",
    order=[nomes_meses[m] for m in range(1, 13)],
    ax=ax,
)
ax.set_title("Quantidade de registros por m\u00eas do evento")
ax.set_ylabel("Quantidade de registros")
ax.set_xlabel("M\u00eas")
ax.legend(title="Classe")

fig.tight_layout()
fig.savefig(DIR_FIGURAS / "grafico3_registros_por_mes.png", dpi=150)
plt.show()
print(f"Salvo em {DIR_FIGURAS / 'grafico3_registros_por_mes.png'}")

Salvo em C:\Users\Virtron\Documents\sergio pessoal\atlas-digital\reports\figures\grafico3_registros_por_mes.png


C:\Users\Virtron\AppData\Local\Temp\ipykernel_18836\4067419575.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Leitura:** EXCESSO concentra seus registros nos meses de verao (janeiro a marco), com pico em janeiro (3.867 registros). FALTA se distribui de forma mais espalhada, com picos em abril (3.972) e janeiro (3.931). A sazonalidade oposta torna o mes do evento um atributo util para a classificacao.

---

### 5. Gráfico 4: proporção de cada classe por região

Distribuição percentual de FALTA e EXCESSO dentro de cada região do Brasil.

In [7]:
ct_regiao = pd.crosstab(df["regiao"], df["alvo"], normalize="index") * 100
ct_regiao = ct_regiao.sort_values("FALTA", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ct_regiao.plot(kind="bar", stacked=True, ax=ax, color=["#d62728", "#1f77b4"])
ax.set_title("Propor\u00e7\u00e3o de FALTA e EXCESSO por regi\u00e3o")
ax.set_ylabel("% dos registros da regi\u00e3o")
ax.set_xlabel("Regi\u00e3o")
ax.legend(title="Classe")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Anotar percentuais
for i, regiao in enumerate(ct_regiao.index):
    falta_pct = ct_regiao.loc[regiao, "FALTA"]
    excesso_pct = ct_regiao.loc[regiao, "EXCESSO"]
    ax.text(i, falta_pct / 2, f"{falta_pct:.0f}%", ha="center", va="center", fontsize=9, fontweight="bold")
    ax.text(i, falta_pct + excesso_pct / 2, f"{excesso_pct:.0f}%", ha="center", va="center", fontsize=9, fontweight="bold")

fig.tight_layout()
fig.savefig(DIR_FIGURAS / "grafico4_classe_por_regiao.png", dpi=150)
plt.show()
print(f"Salvo em {DIR_FIGURAS / 'grafico4_classe_por_regiao.png'}")

Salvo em C:\Users\Virtron\Documents\sergio pessoal\atlas-digital\reports\figures\grafico4_classe_por_regiao.png


C:\Users\Virtron\AppData\Local\Temp\ipykernel_18836\2513309863.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Leitura:** No Nordeste, 85% dos registros sao FALTA, refletindo a recorrencia de secas na regiao. No Norte e no Centro-Oeste, a situacao se inverte: 68% e 63% de EXCESSO, respectivamente. O Sul fica equilibrado (51% FALTA, 49% EXCESSO) e o Sudeste pende levemente para EXCESSO (55%). A regiao tem relacao clara com o alvo.

---

### 6. Tabela: % de valores > 0 por classe para todas as colunas numéricas

Para cada coluna numérica candidata, calculamos o percentual de registros com valor maior que zero em cada classe. A tabela é ordenada pela diferença absoluta entre as classes, destacando os atributos que mais discriminam FALTA e EXCESSO.

In [8]:
# Selecionar colunas numericas, excluindo vazamento e identificacao
colunas_excluir_tabela = [
    "alvo", "mes_evento",
    "tipologia", "Cod_Cobrade", "Cod_IBGE_Mun",
    "DH_AFETADOS_SECA_ESTIAGEM",
]

numericas = [
    c for c in df.select_dtypes(include="number").columns
    if c not in colunas_excluir_tabela
]

linhas = []
for col in numericas:
    pct_falta = (df.loc[df["alvo"] == "FALTA", col].fillna(0) > 0).mean() * 100
    pct_excesso = (df.loc[df["alvo"] == "EXCESSO", col].fillna(0) > 0).mean() * 100
    diferenca = abs(pct_excesso - pct_falta)
    linhas.append({
        "coluna": col,
        "pct_gt0_FALTA": round(pct_falta, 2),
        "pct_gt0_EXCESSO": round(pct_excesso, 2),
        "diferenca_pp": round(diferenca, 2),
    })

df_tabela = (
    pd.DataFrame(linhas)
    .sort_values("diferenca_pp", ascending=False)
    .reset_index(drop=True)
)

# Salvar CSV
caminho_csv = DIR_METRICAS / "diferenca_por_classe.csv"
df_tabela.to_csv(caminho_csv, index=False)
print(f"Tabela salva em {caminho_csv}")
print()
display(df_tabela)

Tabela salva em C:\Users\Virtron\Documents\sergio pessoal\atlas-digital\reports\metrics\diferenca_por_classe.csv



,coluna,pct_gt0_FALTA,pct_gt0_EXCESSO,diferenca_pp
0,DM_total_danos_materiais,1.60,54.72,53.12
1,DM_Obras de Infra Valor,1.04,48.49,47.45
2,DH_DESALOJADOS,0.60,44.55,43.94
3,DM_Uni Habita Danificadas,0.14,39.86,39.72
4,DM_Uni Habita Valor,0.24,38.72,38.49
5,DM_Obras de Infra Danificadas,0.36,35.29,34.93
6,DH_DESABRIGADOS,0.25,31.77,31.52
7,PEPL_Abast de água pot(R$),44.58,13.97,30.60
8,PEPR_Agricultura (R$),58.37,32.64,25.73
9,DM_Obras de Infra Destruidas,0.12,24.20,24.08


**Leitura:** O total de danos materiais lidera com 53,1 pp de diferenca (54,7% em EXCESSO vs. 1,6% em FALTA). Em seguida vem infraestrutura (47,5 pp), desalojados (43,9 pp) e moradias danificadas (39,7 pp). Do lado de FALTA, abastecimento de agua (30,6 pp), agricultura (25,7 pp) e pecuaria (23,8 pp) se destacam. As variaveis com mais de 10 pp de diferenca formam o nucleo dos preditores candidatos.